In [0]:
%sql
use catalog claims;

In [0]:
#Call out all the parameters for specific study
TAB='KEYTRUDA'
VAR='MAR25'
dfs={}
COM=['KEYTRUDA','OPDIVO','IMFINZI','TECENTRIQ','BAVENCIO','LIBTAYO','YERVOY','CABOMETYX','LENVIMA','TRODELVY','PADCEV']
cfg={}
cfg["VAR"]='MAR25'
cfg["ST_DATE"]='2023-03-01'
cfg["CMP_ST_DATE"]='2024-03-01'
cfg["END_DATE"]='2025-03-31'

In [0]:
from pyspark.sql.functions import *
dfs[f"{TAB}_MD_{VAR}"]=spark.read.format("csv").option("header", True).load(
    "s3://athena-input123/dev_db/drug_universe/drug_universe.csv").filter(col("DRUG_NAM").isin(COM))
display(dfs[f"{TAB}_MD_{VAR}"])

In [0]:
from pyspark.sql.functions import *
dfs[f"{TAB}_MD_{VAR}"] = spark.read.format("delta") \
    .load("s3://athena-input123/dev_db/drug_universe_delta/")

display(dfs[f"{TAB}_MD_{VAR}"])

In [0]:
dfs["rx_CLAIMS"] = spark.read.format("parquet") \
    .load("s3://athena-input123/dev_db/prod/rx_claim_master_parquet/") 

display(dfs["rx_CLAIMS"])

In [0]:
dfs["rx_CLAIMS"].select(min(col("RX_FILL_DTE")),max(col("RX_FILL_DTE"))).show()

In [0]:
dfs["rx_CLAIMS"].count()

In [0]:
dfs["person_data"] = spark.read.format("parquet") \
    .load("s3://athena-input123/dev_db/prod/real_person/real_person_master_lvl2_parquet/")

display(dfs["person_data"])

In [0]:
df_pxclaims = spark.read.format("parquet") \
    .load("s3://athena-input123/dev_db/prod/procedure_lvl_pol/procedure_lvl2_pol_large_parquet/")

display(df_pxclaims)

In [0]:
df_pxclaims.select(min(col("PRCDR_DTE")),max(col("PRCDR_DTE"))).show()

In [0]:
# files = dbutils.fs.ls("s3://athena-input123/dev_db/prod/rx_claim_master_parquet/")

# total_bytes = 0
# for partition in files:
#     if partition.name != "_SUCCESS":
#         try:
#             inner_files = dbutils.fs.ls(partition.path)
#             total_bytes += __builtins__.sum(f.size for f in inner_files)
#         except:
#             pass

# total_mb = total_bytes / (1024**2)
# total_gb = total_bytes / (1024**3)

# print(f"Parquet size : {total_mb:.2f} MB / {total_gb:.2f} GB")
# print(f"S3 cost/run  : ${total_gb * 0.09:.4f}")
# print(f"Cost/month (3 runs/day) : ${total_gb * 0.09 * 3 * 30:.4f}")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, trim, to_date, lit

# Define the window specification
rx_window = Window.partitionBy(
    "EDW_PRSN_REAL_GID",
    "RX_FILL_DTE",
    "DS_DRUG_GID",
    "DS_PHMCY_GID",
    "DRUG_DSPND_MTRC_DEC_QTY",
    "NEW_REFIL_CDE"
).orderBy(col("TXN_GID").desc())

# Alias each dataframe to reference columns explicitly
df_rx_alias = dfs["rx_CLAIMS"].alias("rx")
df_md_alias = dfs[f"{TAB}_MD_{VAR}"].alias("md")
df_ptnt_alias = dfs["person_data"].alias("ptnt")

# Build the dataframe with all transformations
dfs[f"PP_{TAB}_RX_{VAR}"] = (
    df_rx_alias
    # Apply filters first (before joins for better performance)
    .filter(col("rx.RX_FILL_DTE") >= "2023-03-01")
    .filter(col("rx.RX_FILL_DTE") <= "2025-03-31")
    .filter(col("rx.EDW_CLAIM_CURR_IND") == "Y")
    .filter(col("rx.EDW_CURR_FINAL_STUS_CDE") == "P")
    .filter(col("rx.RPTG_PRC_REL_GID").isNotNull())
    .filter(col("rx.VNDR_ACCT_ID") != "2")
    # Inner join with drug master data
    .join(
        df_md_alias,
        col("rx.DS_DRUG_GID") == col("md.DRUG_GNRTD_ID"),
        "inner"
    )
    # Left join with person master
    .join(
        df_ptnt_alias,
        col("rx.EDW_PRSN_REAL_GID") == col("ptnt.PRSN_REAL_GID"),
        "left"
    )
    # Add row number
    .withColumn("ROW_NBR", row_number().over(rx_window))
    # Filter to keep only first row per partition
    .filter(col("ROW_NBR") == 1)
    # Select and rename columns to match SQL output - explicitly specify table aliases
    .select(
        trim(col("rx.TXN_GID").cast("string")).alias("CLAIM_GID"),
        col("rx.TXN_GID"),
        col("rx.EDW_PRSN_REAL_GID").alias("PTNT_GID"),
        col("rx.RPTG_PRC_REL_GID").alias("PRC_REL_GID"),
        col("rx.RX_FILL_DTE"),
        col("md.DRUG_NAM"),
        col("md.DRUG_STRGH"),
        col("rx.DAYS_SPLY_NBR").alias("SPLY"),
        col("rx.DRUG_DSPND_MTRC_DEC_QTY").alias("QTY"),
        col("rx.NEW_REFIL_CDE"),
        col("ROW_NBR")
    )
)

# Display the result
display(dfs[f"PP_{TAB}_RX_{VAR}"])

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, trim, to_date, lit

px_window=Window.partitionBy("PRSN_REAL_GID","PRCDR_DTE","PRCDR_CDE","PRCDR_MDFR_1_CDE","PRCDR_MDFR_2_CDE").orderBy(col("TXN_GID").desc())
df_px_alias = df_pxclaims.alias("px")
df_md_alias = dfs[f"{TAB}_MD_{VAR}"].alias("md")
#df_prcdrclaims_alias = df_prcdrclaims.alias("prcdr")


drug_nam_list=[row.DRUG_NAM for row in df_md_alias.select("DRUG_NAM").distinct().collect()]

dfs[f"PP_{TAB}_PX_{VAR}"] = (
            df_px_alias.join(df_md_alias,df_px_alias.DS_DRUG_GID==df_md_alias.DRUG_GNRTD_ID,"left").
            filter(col("px.PRCDR_CDE").isin('S0116','J9062','C9418',
            'C9415','J7527','J8561','J9015','C9491','J9023','J9035',
            'Q5107','Q5118','J9045','J9060','J9000','J9001','J9196',
            'J9201','C9284','J9199','J9198','J9228','C9453','J9299',
            'C9027','J9271','C9239','J9330') | 
            (col("px.PRCDR_CDE").isin('C9399','J9999','J3490','J3590','J8499','J8999','T1502',
            'S5000','S5001') & col("md.DRUG_NAM").isin(drug_nam_list))).
            filter(~col("VNDR_ACCT_ID").isin('2','3','17')).
            filter(col("SRVC_FROM_DTE") >="2023-03-01").
            filter(col("SRVC_FROM_DTE") <="2025-03-31").
            withColumn("ROW_NUMBER",row_number().over(px_window)).
            filter(col("ROW_NUMBER")==1).
            withColumn("BUCKET",when(col("px.PRCDR_CDE").isin('C9027','J9271') | (col("px.PRCDR_CDE").
            isin('C9399','J9999','J3490','J3590','J8499','J8999','T1502','S5000','S5001') & col("DRUG_NAM")=="KEYTRUDA"),"1").
            otherwise("2")).
            withColumn("COMBO_PRCTR_GID",when(col("PRMRY_PRCTR_GID").isNotNull(),col("PRMRY_PRCTR_GID")).when((col("PRMRY_PRCTR_GID").isNull() & col("RFRNG_PRCTR_GID").isNotNull()),col("RFRNG_PRCTR_GID")).otherwise(col("PRMRY_PRCTR_GID"))).
            select(trim(col("px.TXN_GID").cast("string")).alias("CLAIM_GID"),
                   col("px.TXN_GID"),
                   col("px.PRMRY_PRCTR_GID"),
                   col("px.PRSN_REAL_GID").alias("PTNT_GID"),
                   col("px.SRVC_FROM_DTE"),
                   col("px.PRCDR_CDE"),
                   col("px.PRCDR_DTE"),
                   col("md.PDC_NDC_NBR"),
                   col("md.DRUG_NAM"))
                   )
display(dfs[f"PP_{TAB}_PX_{VAR}"])           

In [0]:
dfs[f"PP_{TAB}_RXPX_{VAR}"]=dfs[f"PP_{TAB}_RX_{VAR}"].select(lit("C0000").alias('PRCDR_CDE'),
                                 col("CLAIM_GID"),
                                 col("PRC_REL_GID"),
                                 col("PTNT_GID"),
                                 col("RX_FILL_DTE"),
                                 col("RX_FILL_DTE").alias("PDATE"),
                                 col("DRUG_NAM")).unionByName(
                                 dfs[f"PP_{TAB}_PX_{VAR}"].select(
                                 col("PRCDR_CDE"),
                                 col("CLAIM_GID"),
                                 col("PRMRY_PRCTR_GID").alias("PRC_REL_GID"),
                                 col("PTNT_GID"),
                                 col("SRVC_FROM_DTE").alias("RX_FILL_DTE"),
                                 col("SRVC_FROM_DTE").alias("PDATE"),   
                                 col("DRUG_NAM")
                                 )).filter(col("DRUG_NAM").isNotNull())

In [0]:
display(dfs[f"PP_{TAB}_RXPX_{VAR}"].where(col("DRUG_NAM").isNotNull()))

In [0]:
#PDM DATA
dfs[f"PP_{TAB}_RXPX_{VAR}"].select(min(col("PDATE")),max(col("PDATE"))).show()

In [0]:
def months_between_dates(start,end):
    return (end.year-start.year)*12 + (end.month-start.month)+1

In [0]:
#Call out rest of the parameters
from datetime import datetime
pre_start=datetime.strptime(cfg["ST_DATE"],"%Y-%m-%d")
cmp_start=datetime.strptime(cfg["CMP_ST_DATE"],"%Y-%m-%d")
end=datetime.strptime(cfg["END_DATE"],"%Y-%m-%d")
cfg["TOT_MONTH"]=months_between_dates(pre_start,end)
cfg["PRE_PERIOD"]=12
cfg["POST_PERIOD"]=months_between_dates(cmp_start,end)
cfg["NOH"]=cfg["POST_PERIOD"]


In [0]:
key = f"PP_{TAB}_RXPX_{VAR}"

# Save as Unity Catalog table
dfs[key].write.format("delta").mode("overwrite").saveAsTable(f"claims.default.{key}")

# Reload in any future session
dfs[key] = spark.read.table(f"claims.default.{key}")

In [0]:
from pyspark.sql.functions import *
key = f"PP_{TAB}_RXPX_{VAR}"
dfs[key] = spark.read.table("claims.default.PP_KEYTRUDA_RXPX_MAR25")

In [0]:
%sql
select distinct _metadata.file_path  from claims.default.PP_KEYTRUDA_RXPX_MAR25;

In [0]:
def months_between_dates(start,end):
    return (end.year-start.year)*12 + (end.month-start.month)+1

In [0]:
#Call out rest of the parameters
from datetime import datetime
pre_start=datetime.strptime(cfg["ST_DATE"],"%Y-%m-%d")
cmp_start=datetime.strptime(cfg["CMP_ST_DATE"],"%Y-%m-%d")
end=datetime.strptime(cfg["END_DATE"],"%Y-%m-%d")
cfg["TOT_MONTH"]=months_between_dates(pre_start,end)
cfg["PRE_PERIOD"]=12
cfg["POST_PERIOD"]=months_between_dates(cmp_start,end)
cfg["NOH"]=cfg["POST_PERIOD"]
cfg["MEA_BUCKET"]=1


In [0]:
def months_creation_function(df,key):
    dfs["df_month"]=dfs[key].withColumn("MONTH",abs(months_between(lit(cfg["ST_DATE"]),col("PDATE")).cast("int")-1))
    dfs["df_month"]=dfs["df_month"].withColumn("RELGID",lpad(col("PRC_REL_GID"),10,"0")).drop("PRC_REL_GID")
    dfs["df_month"]=dfs["df_month"].withColumn("BUCKET",when(col("DRUG_NAM")=='KEYTRUDA',"1").otherwise("2"))
    return dfs["df_month"]


In [0]:
dfs["df_month"]=months_creation_function(dfs,key)

In [0]:
from pyspark.sql.window import Window
window=Window.partitionBy("PTNT_GID","RELGID","DRUG_NAM","PDATE","CLAIM_GID").orderBy("PTNT_GID","RELGID","DRUG_NAM","PDATE","CLAIM_GID")
dfs["df_unique"]=dfs["df_month"].withColumn("row_num",row_number().over(window)).filter(col("row_num")==1)
dfs["df_duplicate"]=dfs["df_month"].withColumn("row_num",row_number().over(window)).filter(col("row_num")==2)

In [0]:
df_bucket={}
cfg["BUCKET"]=dfs["df_unique"].select("BUCKET").distinct().count()
for i in range(cfg["BUCKET"]):
    df_bucket[f"df_{i+1}"]=dfs["df_unique"].filter(col("BUCKET")==i+1).groupBy("RELGID").pivot("MONTH").agg(count("CLAIM_GID"))
    col_name=["RELGID" if j=="RELGID" else "B0"+str(i+1)+"TRX"+str(j) for j in df_bucket[f"df_{i+1}"].columns]
    df_bucket[f"df_{i+1}"]=df_bucket[f"df_{i+1}"].toDF(*col_name)

In [0]:
from functools import reduce
for key in df_bucket.keys():
    df_bucket[key]=df_bucket[key].withColumn("RELGID",col("RELGID").cast("string"))
df_bucket_list=[df_bucket[i] for i in df_bucket.keys()]
df_bucket_combine=reduce(lambda x1,x2: x1.join(x2,how="full_outer",on="RELGID"), df_bucket_list)
df_bucket_combine=df_bucket_combine.fillna(0)

In [0]:
# creating market varibale
for i in range(1,cfg["TOT_MONTH"]+1):
    sum_string=["B0"+str(k)+"TRX"+str(i) for k in range(1,cfg["BUCKET"]+1)]
    df_bucket_combine=df_bucket_combine.withColumn(f"MKTTRX"+str(i),expr("+".join(sum_string)))
dfs["df_fin"]=df_bucket_combine
    

In [0]:
#display(df_bucket_combine),df_bucket_combine.count()

In [0]:
#Creating state 
spec={}
dfs["df_fin"]=dfs["df_fin"].withColumn("RANDOM",(rand()*8+1).cast("int")).\
                    withColumn("STATE",when(col("RANDOM")==1,"AB").\
                        when(col("RANDOM")==2,"BC").\
                            when(col("RANDOM")==3,"CD").\
                                when(col("RANDOM")==4,"DE").\
                                    when(col("RANDOM")==5,"EF").\
                                        when(col("RANDOM")==6,"GH").\
                                            when(col("RANDOM")==7,"HI").\
                                                when(col("RANDOM")==8,"JK")).drop("RANDOM")
                    


In [0]:
df_test= dfs["df_fin"].select("RELGID").distinct().sample(fraction=0.10, seed=42)

In [0]:
#Adding cohort
spec["df_test"]=df_test.withColumn("RANDOM",(rand()*13+1).cast("int")).\
                    withColumn("COHORT",when(col("RANDOM")==1,"A").\
                        when(col("RANDOM")==2,"B").\
                            when(col("RANDOM")==3,"C").\
                                when(col("RANDOM")==4,"D").\
                                    when(col("RANDOM")==5,"E").\
                                        when(col("RANDOM")==6,"F").\
                                            when(col("RANDOM")==7,"G").\
                                                when(col("RANDOM")==8,"H").\
                                                    when(col("RANDOM")==9,"I").\
                                                        when(col("RANDOM")==10,"J").\
                                                            when(col("RANDOM")==11,"K").\
                                                                when(col("RANDOM")==12,"L").\
                                                                    when(col("RANDOM")==13,"M")).\
                                                                        drop("RANDOM").select("RELGID","COHORT")

In [0]:
# Adding Speciality
dfs["df_fin"]=dfs["df_fin"].withColumn("RANDOM",(rand()*5+1).cast("int")).\
                                   withColumn("SPEC",when(col("RANDOM")==1,"ONCO").\
                                       when(col("RANDOM")==2,"RADIO").\
                                           when(col("RANDOM")==3,"GEN").\
                                               when(col("RANDOM")==4,"NPPA").\
                                                   when(col("RANDOM")==5,"DEA").\
                                                       when(col("RANDOM")==6,"OTHER")).\
                                                           drop("RANDOM")
                                             


In [0]:
# from pyspark.sql import functions as F

# months = []
# for ym in range(2303, 2504):
#     yy = ym // 100
#     mm = ym % 100
#     if 1 <= mm <= 12:
#         months.append(f"LD{ym}")

# df = spark.range(10000, 14000).select(
#     F.lpad(F.col("id").cast("string"), 10, "0").alias("RELGID")
# )

# for m in months:
#     df = df.withColumn(m, F.lit(1))

# df_detail=df

In [0]:
from pyspark.sql import functions as F
months = []
for ym in range(2303, 2504):
    yy = ym // 100
    mm = ym % 100
    if 1 <= mm <= 12:
        months.append(f"LD{ym}")

df = spark.range(10000, 14000).select(
    F.lpad(F.col("id").cast("string"), 10, "0").alias("RELGID")
)

for m in months:
    df = df.withColumn(m, (F.rand()*2).cast("int"))  # randomly 0 or 1

df_detail = df

In [0]:
df_detail=df_detail.toDF(*["D"+str(cfg["TOT_MONTH"]+1-d[0]) if d[1]!='RELGID' else d[1] for d in enumerate(df_detail.columns)])

In [0]:
dfs["df_final"]=dfs["df_fin"].join(df_detail,on="RELGID",how="left")
dfs["df_final"]=dfs["df_final"].fillna(0)
dfs["df_final"].write.format("delta").mode("overwrite").saveAsTable(f"claims.default.df_final")
  # materialize

In [0]:
%sql
select distinct _metadata.file_path from claims.default.df_final;

In [0]:
spark.sql("SHOW TABLES IN claims.default").show(truncate=False)

In [0]:
%sql
DESCRIBE DETAIL claims.default.df_control;

In [0]:
tables = spark.sql("SHOW TABLES IN claims.default").collect()

for row in tables:
    table_name = row["tableName"]
    location = spark.sql(f"DESCRIBE DETAIL claims.default.{table_name}") \
                    .collect()[0]["location"]
    print(f"{table_name} → {location}")

In [0]:
cfg.keys()

In [0]:

# stats={}
# for coh in range(1,cfg["NOH"]): 
#     h=2 #to compenstate for sliding backward and make less clutter to below code
#     q1=["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h,cfg["POST_PERIOD"]+4-coh+h)]
#     q2=["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h+4,cfg["POST_PERIOD"]+8-coh+h)]
#     q3=["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h+8,cfg["POST_PERIOD"]+12-coh+h)]
#     q4=["MKTTRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h,cfg["POST_PERIOD"]+cfg["PRE_PERIOD"]+h-coh)]
#     q5=["D"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h,cfg["POST_PERIOD"]+cfg["PRE_PERIOD"]+h-coh)]
#     sum_q=q1+q2+q3+q4+q5
#     stats[f"cohort_{coh}"]=dfs["df_final"].select(*sum_q)
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("q1",expr("+".join(q1)))
#     q1_avg,q1_std=stats[f"cohort_{coh}"].select(mean(col("q1")),stddev(col("q1"))).collect()[0] 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("z1",(col("q1")-lit(q1_avg))/lit(q1_std))      
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("q2",expr("+".join(q2)))
#     q2_avg,q2_std=stats[f"cohort_{coh}"].select(mean(col("q2")),stddev(col("q2"))).collect()[0] 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("z2",(col("q2")-lit(q2_avg))/lit(q2_std)) 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("q3",expr("+".join(q3)))
#     q3_avg,q3_std=stats[f"cohort_{coh}"].select(mean(col("q3")),stddev(col("q3"))).collect()[0] 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("z3",(col("q3")-lit(q3_avg))/lit(q3_std))     
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("q4",expr("+".join(q4)))
#     q4_avg,q4_std=stats[f"cohort_{coh}"].select(mean(col("q4")),stddev(col("q4"))).collect()[0] 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("z4",(col("q4")-lit(q4_avg))/lit(q4_std)) 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("q5",expr("+".join(q5)))
#     q5_avg,q5_std=stats[f"cohort_{coh}"].select(mean(col("q5")),stddev(col("q5"))).collect()[0] 
#     stats[f"cohort_{coh}"]=stats[f"cohort_{coh}"].withColumn("z5",(col("q5")-lit(q5_avg))/lit(q5_std)) 

#     #stats[f"cohort_{coh}_q3"].show()
#     #break
    


In [0]:
dfs["df_final"]=spark.read.table("claims.default.df_final")
display(dfs["df_final"])

In [0]:
stats = {}
dfs["df_final"]=spark.read.table("claims.default.df_final")
for coh in range(1, cfg["NOH"]+1):
    h = 2
    q1 = ["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h, cfg["POST_PERIOD"]+4-coh+h)]
    q2 = ["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h+4, cfg["POST_PERIOD"]+8-coh+h)]
    q3 = ["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h+8, cfg["POST_PERIOD"]+12-coh+h)]
    q4 = ["MKTTRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h, cfg["POST_PERIOD"]+cfg["PRE_PERIOD"]+h-coh)]
    q5 = ["D"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h, cfg["POST_PERIOD"]+cfg["PRE_PERIOD"]+h-coh)]
    avgBPre=["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"]-coh+h, cfg["POST_PERIOD"]+cfg["PRE_PERIOD"]+h-coh)]
    avgMPre=["MKTTRX"+ str(k) for k in range(cfg["POST_PERIOD"]-coh+h, cfg["POST_PERIOD"]+cfg["PRE_PERIOD"]+h-coh)]
    avgBPost=["B0"+str(cfg["MEA_BUCKET"])+"TRX"+str(k) for k in range(cfg["POST_PERIOD"],1,-1)] 
    slope_var=f"({avgBPre[0]}-{avgBPre[-1]}/{len(avgBPre)})" 

    df = dfs["df_final"].select(*(q1+q2+q3+q4+q5+avgBPost+avgMPre+["RELGID"])) \
        .withColumn("q1", expr("+".join(q1))) \
        .withColumn("q2", expr("+".join(q2))) \
        .withColumn("q3", expr("+".join(q3))) \
        .withColumn("q4", expr("+".join(q4))) \
        .withColumn("q5", expr("+".join(q5))) \
        .withColumn("avgBPost",expr("+".join(avgBPre))/len(avgBPre))\
        .withColumn("avgMPre",expr("+".join(avgMPre))/len(avgMPre))\
        .withColumn("avgBPre",expr("+".join(avgBPost))/len(avgBPost))\
        .withColumn("SLOPE",expr(slope_var))
        


    # Single collect for all 10 stats
    agg_row = df.select(
        mean("q1").alias("q1_avg"), stddev("q1").alias("q1_std"),
        mean("q2").alias("q2_avg"), stddev("q2").alias("q2_std"),
        mean("q3").alias("q3_avg"), stddev("q3").alias("q3_std"),
        mean("q4").alias("q4_avg"), stddev("q4").alias("q4_std"),
        mean("q5").alias("q5_avg"), stddev("q5").alias("q5_std"),
    ).collect()[0]

    df = df.withColumn("z1", (col("q1")-lit(agg_row["q1_avg"]))/lit(agg_row["q1_std"])) \
           .withColumn("z2", (col("q2")-lit(agg_row["q2_avg"]))/lit(agg_row["q2_std"])) \
           .withColumn("z3", (col("q3")-lit(agg_row["q3_avg"]))/lit(agg_row["q3_std"])) \
           .withColumn("z4", (col("q4")-lit(agg_row["q4_avg"]))/lit(agg_row["q4_std"])) \
           .withColumn("z5", (col("q5")-lit(agg_row["q5_avg"]))/lit(agg_row["q5_std"]))

    stats[f"cohort_{coh}"] = df.select("RELGID","z1","z2","z3","z4","z5","avgBPost","avgBPre","avgMPre","SLOPE")
    stats[f"cohort_{coh}"].write.mode("overwrite").saveAsTable(f"claims.default.cohort_{coh}")
    

In [0]:
stats.keys()

In [0]:
cfg["ENUM"]=2
slmatch={}
matching_var=["RELGID","STATE","SPEC","z1","z2","z3","z4","z5","avgBPre","avgMpre","avgBPost","SLOPE"]
cohort_maps={"1":"A","2":"B","3":"C","4":"D","5":"E","6":"F","7":"G","8":"H","9":"I","10":"J","11":"K","12":"L","13":"M","14":"N","15":"O","16":"P","17":"Q","18":"R","19":"S"}
#df_control=dfs["df_final"].select(*matching_var).join(spec["df_test"],how="left_anti",on="RELGID").toDF(*["c"+cols for cols in matching_var])
df_control=dfs["df_final"].select("RELGID","STATE","SPEC").join(spec["df_test"],how="left_anti",on="RELGID")
df_control.write.mode("overwrite").saveAsTable("claims.default.df_control")
df_control=spark.read.table("claims.default.df_control")
#df_test=dfs["df_final"].join(spec["df_test"],how="inner",on="RELGID").select(*matching_var,"COHORT")
df_test=dfs["df_final"].select("RELGID","STATE","SPEC").join(spec["df_test"],how="inner",on="RELGID")
df_test.write.mode("overwrite").saveAsTable("claims.default.df_test")
df_test=spark.read.table("claims.default.df_test")
for i in range(1,cfg["NOH"]+1):
    stats[f"cohort_{i}"]=spark.read.table(f"claims.default.cohort_{i}")
    df_test1=df_test.join(stats[f"cohort_{i}"],how="inner",on="RELGID").filter(df_test.COHORT==cohort_maps[str(i)])
    df_control1=df_control.join(stats[f"cohort_{i}"],how="inner",on="RELGID")
    control_cols=["c"+colm for colm in df_control1.columns]
    df_control1=df_control1.toDF(*control_cols)
    df_match1=df_test1.join(df_control1,(col('STATE')==col('cSTATE')) & (col('SPEC')==col('cSPEC')), how="inner")
    df_match2=df_match1.withColumn("TRXdiff",abs(col("cavgBpre")-col("avgBpre")))\
                            .withColumn("MKTdiff",abs(col("cavgMpre")-col("avgMpre")))\
                            .withColumn("SLOPEdiff",abs(col("SLOPE")-col("cSLOPE")))\
                            .withColumn("distance",sqrt((col("z1")-col("cz1"))**2+(col("z2")-col("cz2"))**2+(col("z3")-col("cz3"))**2+(col("z4")-col("cz4"))**2+(col("z5")-col("cz5"))**2))
    df_eumatched=df_match2.filter(col("distance")<=15)
    #df_match_sort=df_match2.orderBy("RELGID","TRXdiff","MKTdiff","SLOPEdiff")
    #df_match_iter1=df_match_sort.filter(col("distance")<=cfg["ENUM"])
    test_window=Window.partitionBy("RELGID").orderBy("distance", "TRXdiff", "MKTdiff", "SLOPEdiff")
    slmatch[f"df_match_iter2_ch{i}"]=df_eumatched.withColumn("ronum",row_number().over(test_window)).filter(col("ronum")==1)
    slmatch[f"df_match_iter2_ch{i}"].write.mode("overwrite").saveAsTable(f"claims.default.df_match_iter2_ch{i}")
    slmatch[f"df_match_iter2_ch{i}"]=spark.read.table(f"claims.default.df_match_iter2_ch{i}")
    df_control=df_control.join(slmatch[f"df_match_iter2_ch{i}"].select("cRELGID"),col("RELGID")==col("cRELGID"),how="left_anti")
    df_control.write.mode("overwrite").saveAsTable("claims.default.df_control")
    df_control = spark.read.table("claims.default.df_control")
    

In [0]:
k=[]
test_hcps=[k.append(i[1].select("RELGID").collect()) for i in slmatch.items()]

In [0]:
print(test_hcps)

In [0]:
slmatch.keys()